# Sampling MAPbI$_3$ + H$_2$O structures with SOAP and FPS

This notebook builds a small set of candidate atomic structures, describes them with **SOAP descriptors**, and then selects a diverse subset with **Farthest Point Sampling (FPS)**.

## What you will learn
- how to build a crystal structure and a supercell with ASE,
- how to add water molecules with a simple random-placement strategy,
- how SOAP descriptors encode local atomic environments,
- how a distance matrix can be used to select diverse structures.

## Strategy of the notebook
1. Build a `2×2×2` supercell of MAPbI$_3$.
2. Generate 20 candidate structures by adding 2 water molecules in random positions/orientations.
3. Compute SOAP descriptors for each structure.
4. Convert the descriptors into a distance matrix.
5. Select 10 diverse structures with FPS.


In [ ]:
%load_ext aiida
%aiida

In [ ]:
# Numerical tools
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.spatial.transform import Rotation

# Atomic structures and visualization
from ase.spacegroup import crystal
from ase import neighborlist
from ase.build import molecule
from itertools import product
import aiidalab_widgets_base as awb
from surfaces_tools.widgets import editors
import ipywidgets as ipw

# SOAP representation
from rascal.representations import SphericalInvariants

In [ ]:
structure_selector = awb.StructureManagerWidget(
    importers=[],
    editors=[],
    storable=True,
    node_class="StructureData",
)


## Helper functions

The next cell defines a few small utility functions that we will reuse later.

- `get_systems_tag`: tells us which atoms belong to which structure.
- `avg_soaps`: averages atomic SOAP vectors into one vector per structure.
- `get_dist_mat`: converts descriptors into a pairwise distance matrix.
- `get_kernel_mat`: shows how one could build a similarity matrix instead.


In [ ]:
def get_systems_tag(frames):
    """Return an array that labels each atom with the index of its parent structure.

    Example:
        if frames contains 3 structures with 5, 7, and 6 atoms,
        the returned array has length 18 and looks like
        [0, 0, 0, 0, 0, 1, 1, ..., 2, 2, 2].
    """
    labels = []
    for i, frame in enumerate(frames):
        labels.extend([i] * len(frame))
    return np.array(labels)


def get_dist_mat(soap_vectors, normalized=True):
    """Build a pairwise Euclidean distance matrix from descriptor vectors.

    Parameters
    ----------
    soap_vectors : array-like, shape (n_structures, n_features)
        One descriptor vector per structure.
    normalized : bool
        If True, divide all distances by the maximum distance so that
        values lie approximately between 0 and 1.
    """
    distance = squareform(pdist(soap_vectors))

    max_val = np.max(distance) if normalized else 1.0
    if max_val == 0:
        max_val = 1.0

    distance_df = pd.DataFrame(distance / max_val)

    # Make the printed matrix easier to read in the notebook
    pd.set_option("display.max_columns", None)
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    return distance_df


def avg_soaps(atom_soap_features, frames):
    """Average atomic SOAP vectors to obtain one SOAP vector per structure."""
    df = pd.DataFrame(atom_soap_features)
    df["structure"] = get_systems_tag(frames)
    return df.groupby("structure").mean().values


def get_kernel_mat(soap_vectors, sigma=0.5):
    """Convert distances into a Gaussian kernel (similarity) matrix.

    This is not essential for the notebook, but it is useful to remember that
    descriptors can be turned into either distances or similarities.
    """
    distance = squareform(pdist(soap_vectors))
    kernel_matrix = np.exp(-(distance ** 2) / (2 * sigma ** 2))
    return pd.DataFrame(kernel_matrix)

## 1. Build the starting MAPbI$_3$ structure

We start from a crystallographic description of the **unit cell** and then repeat it to create a larger supercell.  
A larger cell gives more room for placing water molecules without creating too many artificial interactions.


In [ ]:
a = 8.86574
b = 12.6293
c = 8.57684
alpha = 90
beta = 90
gamma = 90

MAPbI3 = crystal(
    symbols="PbI2NCH4",  # some labels in the source table are slightly misleading
    basis=[
        (0.5,    0.0,   0.0),
        (0.4842, 0.25, -0.0562),
        (0.1886, 0.0147, 0.1844),
        (0.9421, 0.75,  0.0297),
        (0.9372, 0.25,  0.0575),
        (0.9372, 0.25,  0.1874),
        (0.8661, 0.1701, 0.0290),
        (0.1275, 0.1891,-0.0085),
        (0.9543, 0.750, 0.1459),
    ],
    spacegroup=62,
    cellpar=[a, b, c, alpha, beta, gamma],
)

## Step 2: Create the 2×2×2 supercell

This is the host structure into which water molecules will be inserted.

In [ ]:
supercell_no_h2o = MAPbI3.repeat((2, 2, 2))
structure_selector.structure = supercell_no_h2o
supercell_no_h2o
structure_selector

## Step 3: Identify the nitrogen atoms

Each water molecule will be inserted near a randomly chosen nitrogen atom belonging to a methylammonium cation.

We also identify the hydrogen atoms bound to nitrogen. This is not strictly required for the insertion algorithm,
but it is often useful when inspecting the local environment.


In [ ]:
cutoffs = neighborlist.natural_cutoffs(supercell_no_h2o)
nl = neighborlist.NeighborList(cutoffs, self_interaction=False, bothways=True)
nl.update(supercell_no_h2o)

all_N = [atom.index for atom in supercell_no_h2o if atom.symbol == "N"]
all_H_of_N = [
    index
    for N in all_N
    for index in nl.get_neighbors(N)[0]
    if supercell_no_h2o[index].symbol == "H"
]
all_nh3 = all_N + all_H_of_N

print(f"Number of nitrogen atoms in the 2x2x2 supercell: {len(all_N)}")
print(f"Number of H atoms bonded to N: {len(all_H_of_N)}")


## Step 4: Define the sampling parameters

- `NH2O`: number of water molecules per sample
- `nsamples`: number of different structures to generate
- `dmin_shell`, `dmax_shell`: allowed distance range between the **oxygen** atom of water and a nitrogen atom
- `min_distance`: minimum allowed distance between **any atom of the trial water molecule** and the atoms already present in the structure
- `max_attempts_per_water`: safety limit for the rejection-sampling loop
- `seed`: random seed for reproducibility

In [ ]:
NH2O = 2
nsamples = 20

dmin_shell = 1.5
dmax_shell = 2.5
min_distance = 1.5

max_attempts_per_water = 1000
seed = 4732


## Step 5: Helper functions


### uniform random rotation in 3D
using `scipy.spatial.transform.Rotation.random()`.

### Strategy used below

For each H$_2$O molecule:

1. choose a random nitrogen atom,
2. choose a random direction in 3D,
3. choose a random O–N distance between `dmin_shell` and `dmax_shell`,
4. apply a random 3D rotation to the water molecule,
5. accept the placement only if all distances from the new water atoms to the atoms already present in the structure are at least `min_distance`.

In [ ]:
def random_unit_vector(rng):
    """Return a random unit vector sampled uniformly on the sphere."""
    vec = rng.normal(size=3)
    return vec / np.linalg.norm(vec)


def make_centered_h2o():
    """
    Build an H2O molecule and translate it so that the oxygen atom sits at the origin.
    This makes rotations and translations easier to apply.
    """
    h2o = molecule("H2O")
    h2o.translate(-h2o.positions[0])  # oxygen is atom 0 in ASE's H2O
    return h2o


def propose_h2o_near_random_n(base_h2o, structure, n_indices, dmin_shell, dmax_shell, rng):
    """
    Create a trial H2O molecule with:
    - oxygen placed at a random distance from a random nitrogen atom,
    - a random orientation in 3D.
    """
    trial_h2o = base_h2o.copy()

    # Choose a random N atom in the pristine MAPbI3 host.
    n_index = int(rng.choice(n_indices))
    n_position = structure.positions[n_index]

    # Choose a random point in the spherical shell around that N atom.
    direction = random_unit_vector(rng)
    distance = rng.uniform(dmin_shell, dmax_shell)
    oxygen_position = n_position + distance * direction

    # Apply a proper random 3D rotation to the water molecule.
    rotation = Rotation.random(random_state=rng)
    trial_h2o.positions = rotation.apply(trial_h2o.positions)

    # Translate so that the oxygen atom reaches the target position.
    trial_h2o.translate(oxygen_position)

    return trial_h2o, n_index


def placement_is_valid(structure, trial_h2o, min_distance):
    """
    Return True if every atom of trial_h2o is at least min_distance away from every atom already
    present in structure. Distances are computed with the minimum-image convention.
    """
    natoms_existing = len(structure)
    natoms_trial = len(trial_h2o)

    combined = structure + trial_h2o

    for i_trial in range(natoms_existing, natoms_existing + natoms_trial):
        for j_existing in range(natoms_existing):
            if combined.get_distance(i_trial, j_existing, mic=True, vector=False) < min_distance:
                return False

    return True


def generate_one_sample(supercell_no_h2o, n_indices, NH2O, dmin_shell, dmax_shell, min_distance, rng, max_attempts_per_water=1000, verbose=True):
    """
    Generate one sample containing NH2O water molecules.
    """
    structure = supercell_no_h2o.copy()
    base_h2o = make_centered_h2o()

    for ih2o in range(NH2O):
        inserted = False

        for attempt in range(1, max_attempts_per_water + 1):
            trial_h2o, n_index = propose_h2o_near_random_n(
                base_h2o=base_h2o,
                structure=structure,
                n_indices=n_indices,
                dmin_shell=dmin_shell,
                dmax_shell=dmax_shell,
                rng=rng,
            )

            if placement_is_valid(structure, trial_h2o, min_distance=min_distance):
                structure += trial_h2o
                inserted = True
                if verbose:
                    print(
                        f"  inserted H2O {ih2o + 1:2d}/{NH2O} "
                        f"after {attempt:3d} attempts near N index {n_index}"
                    )
                break

        if not inserted:
            raise RuntimeError(
                f"Could not place H2O number {ih2o + 1} after {max_attempts_per_water} attempts. "
                "Try decreasing NH2O, increasing the shell size, or relaxing the minimum-distance cutoff."
            )

    structure.wrap()
    return structure


def generate_samples(supercell_no_h2o, n_indices, nsamples, NH2O, dmin_shell, dmax_shell, min_distance, seed=7, max_attempts_per_water=1000, verbose=True):
    """
    Generate nsamples different structures.
    """
    rng = np.random.default_rng(seed)
    samples = []

    for isample in range(nsamples):
        if verbose:
            print(f"Creating sample {isample + 1}/{nsamples}")
        sample = generate_one_sample(
            supercell_no_h2o=supercell_no_h2o,
            n_indices=n_indices,
            NH2O=NH2O,
            dmin_shell=dmin_shell,
            dmax_shell=dmax_shell,
            min_distance=min_distance,
            rng=rng,
            max_attempts_per_water=max_attempts_per_water,
            verbose=verbose,
        )
        samples.append(sample)

    return samples


## Step 6: Generate the statistical ensemble

This cell creates `nsamples` different structures.  
Each structure contains `NH2O` water molecules placed according to the rules above.


In [ ]:
samples = generate_samples(
    supercell_no_h2o=supercell_no_h2o,
    n_indices=all_N,
    nsamples=nsamples,
    NH2O=NH2O,
    dmin_shell=dmin_shell,
    dmax_shell=dmax_shell,
    min_distance=min_distance,
    seed=seed,
    max_attempts_per_water=max_attempts_per_water,
    verbose=True,
)


In [ ]:
# Wrap atoms back into the periodic cell and explicitly activate periodic boundary conditions
for structure in samples:
    structure.wrap()
    structure.pbc = (1, 1, 1)

## STEP 7: Describe each structure with SOAP descriptors

SOAP (**Smooth Overlap of Atomic Positions**) converts each local atomic environment into a numerical vector.  
These vectors are designed so that similar atomic environments have similar numerical representations.

Here we compute SOAP for all atoms and then average the atomic SOAP vectors to obtain **one descriptor per structure**.


In [ ]:
# Hyperparameters for the SOAP representation
hypers = {
    "soap_type": "PowerSpectrum",
    "interaction_cutoff": 5.0,
    "max_radial": 6,
    "max_angular": 6,
    "gaussian_sigma_constant": 0.4,
    "gaussian_sigma_type": "Constant",
    "cutoff_smooth_width": 0.5,
    "radial_basis": "GTO",
    "cutoff_function_type": "ShiftedCosine",
    "cutoff_function_parameters": {"width": 0.5},
    "global_species": [1, 6, 7, 53, 82],  # H, C, N, I, Pb
}

soap = SphericalInvariants(**hypers)

In [ ]:
# Compute the SOAP representation for all structures
soap_rep = soap.transform(samples)

## Step 8: Select diverse structures with Farthest Point Sampling (FPS)

After turning each structure into a descriptor vector, we can compare structures through a **distance matrix**.

FPS works as follows:

1. start from one structure,
2. find the structure that is farthest from the ones already selected,
3. repeat until the desired number of structures is reached.

This is a simple and effective way to keep a **diverse subset**.


In [ ]:
def fps_from_distance_matrix(distance_matrix, n=0, idx=None):
    """Select a diverse subset using Farthest Point Sampling (FPS).

    Parameters
    ----------
    distance_matrix : ndarray, shape (N, N)
        Pairwise distance matrix between structures.
    n : int
        Number of structures to select. If n <= 0, all structures are selected.
    idx : int or None
        Optional starting index. If None, the first point is chosen at random.
    """
    N = distance_matrix.shape[0]

    if n <= 0:
        n = N

    fps_indices = np.zeros(n, dtype=np.int32)
    d = np.zeros(n)

    if idx is None:
        idx = np.random.randint(0, N)

    fps_indices[0] = idx

    # Distance from every point to the first selected point
    d1 = distance_matrix[idx]

    for i in range(1, n):
        # Pick the point that is currently farthest from the selected set
        fps_indices[i] = np.argmax(d1)
        d[i - 1] = np.max(d1)

        # Update the minimum distance to the selected set
        d2 = distance_matrix[fps_indices[i]]
        d1 = np.minimum(d1, d2)

        if d1.max() == 0.0:
            print(f"Only {i} FPS selections are possible.")
            return fps_indices[:i], d[:i]

    return fps_indices

### How the distance matrix is used in FPS

The distance matrix tells us how different each pair of structures is.

During FPS we keep track, for every not-yet-selected structure, of its **distance to the closest selected structure**.  
At each step we pick the structure with the **largest** such distance.

This means we always add the structure that contributes the most new information, in the sense of diversity.


## STEP 9: Build the distance matrix and run FPS


In [ ]:
# Compute one SOAP vector per atom
X = soap_rep.get_features(soap)

# Average atomic SOAP vectors to obtain one descriptor per structure
avg_soap_samples = avg_soaps(X, samples)

# Convert descriptors into a pairwise distance matrix between structures
DistMat = get_dist_mat(avg_soap_samples)

In [ ]:
# DistMat.values converts the pandas DataFrame into a NumPy array
selected_idx = fps_from_distance_matrix(DistMat.values, n=10, idx=None).tolist()

In [ ]:
selected_idx

In [ ]:
# Display the distance matrix
DistMat

### Does the result make sense?

Yes, this is what we expect from FPS.

- The diagonal of `DistMat` is zero because each structure has zero distance from itself.
- The matrix is symmetric because the distance from structure `i` to `j` is the same as from `j` to `i`.
- The selected indices should correspond to structures that are relatively far apart in descriptor space.
- If some structures are very similar, FPS tends to keep only one of them and prefers others that are more different.

In other words, FPS is not selecting the “best” structures in an energetic sense.  
It is selecting the **most diverse** structures according to the SOAP representation.


In [ ]:
sample_slider = ipw.IntSlider(
    value=0,
    min=0,
    max=len(selected_idx) - 1,
    step=1,
    description="Sample:",
    continuous_update=False,
)

def show_sample(sample_index):
    structure_selector.structure = samples[selected_idx[sample_index]]

ipw.interact(show_sample, sample_index=sample_slider);

display(structure_selector)